In [0]:
# Create a function that checks and updates the books_silver layer based on scd type 2
from pyspark.sql.window import Window
def scd_type_2_upsert(microBatchDF, batch):   
    window = Window.partitionBy("book_id").orderBy(F.col("updated").desc())

    deduped_df = (
        microBatchDF
        .withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


    deduped_df.createOrReplaceTempView("updates") 
    
    sql_query = f"""
        MERGE INTO dev.silver.books_silver AS target
        USING (
            -- Stage incoming records with a deterministic checksum
            SELECT
                u.book_id as merge_key, 
                u.book_id,
                u.title,
                u.author,
                u.price,
                u.updated,
                SHA2(
                    CONCAT_WS(
                        '|',
                        COALESCE(u.title, ''),
                        COALESCE(u.author, ''),
                        COALESCE(CAST(u.price AS STRING), '')
                    ),
                    256
                ) AS checksum
            FROM updates u

            UNION ALL

            SELECT
                NULL as merge_key,
                u.book_id,
                u.title,
                u.author,
                u.price,
                u.updated,
                SHA2(
                    CONCAT_WS(
                        '|',
                        COALESCE(u.title, ''),
                        COALESCE(u.author, ''),
                        COALESCE(CAST(u.price AS STRING), '')
                    ),
                    256
                ) AS checksum
            FROM updates u
            JOIN dev.silver.books_silver bs ON u.book_id = bs.book_id
            WHERE bs.current = true
        ) AS src
        ON target.book_id = merge_key
        AND target.current = TRUE

        -- Expire current record only when data actually changed
        WHEN MATCHED
        AND target.checksum <> src.checksum THEN
            UPDATE SET
                target.current   = FALSE,
                target.end_date  = src.updated
            
        -- Insert new records:
        --  1. brand‑new book_id
        --  2. changed records that were expired by the UPDATE above
        WHEN NOT MATCHED THEN
            INSERT (
                book_id,
                title,
                author,
                price,
                checksum,
                current,
                effective_date,
                end_date
            )
            VALUES (
                src.book_id,
                src.title,
                src.author,
                src.price,
                src.checksum,
                TRUE,
                src.updated,
                NULL
            )
        """
    microBatchDF.sparkSession.sql(sql_query);

In [0]:
from pyspark.sql import functions as F
def process_books():
    book_schema = "book_id STRING, title STRING, author STRING, price DOUBLE, updated TIMESTAMP"

    df_books = (
        spark.readStream.table("dev.bookstore_bronze.bookstore_bronze")
            .filter("topic = 'books'")
            .select(
                F.from_json(
                    F.col("value").cast("string"),
                    schema=book_schema

                ).alias('v')
            )
            .select("v.*")
            .writeStream
            .foreachBatch(scd_type_2_upsert)
            .option("checkpointLocation", "/Volumes/dev/landing_zone/kafka_source/checkpoints/books_silver")
            .trigger(availableNow=True)
            .start()    
    )

process_books()